# ENG3014 Communication Channel Simulation

**Kartike** · Communication Systems 3 · 2026/27

A wireless link simulated end to end: **text → bits → modulated carrier → output filter → AWGN channel → demodulator → bits → text**, with the bit error rate (BER) measured over repeated transmissions at 6, 18 and 24 dB SNR.

| Brief requirement | Section |
|---|---|
| Message of more than 2048 bits, string → binary | 1 |
| Modulation format of your choice, baud rate | 2, 3 |
| Output filter: transmitter bandwidth < 20% of the carrier | 4 |
| AWGN channel at 6, 18 and 24 dB | 5 |
| Demodulation and recovery of the message | 6 |
| BER over multiple retransmissions | 7 |
| Creative additions (up to 3 × 10%) | 8 |
| Report figures | 9 |

**Marks:** code and comments 20% · overview and background 20% · results and discussion 20% · presentation 10% · creative additions 30%.

### How this notebook works
Each section gives the requirement, where it is in the slides, a design question, a code cell for you to fill in, and a **CHECK** cell. Write the code, then run the check: it tells you whether the block works without telling you how to write it. Fill in the **Report notes** line as you go, so the reasoning behind each choice is already written when you draft the report.

Slide references: "L4 p.19" is a page of the new lecture PDFs (Prof Wagih). "Old L07 p.49" is Prof Lavery's 2025/26 slides, used for topics the new series hasn't released yet. Anything marked *beyond the slides* needs its own reference.

### The report (3 pages)
1. **Overview** of the simulation approach (a block diagram helps)
2. **Background:** the modulation format, fading model and creative additions, with references
3. **Results:** the key plots, and why you made each choice
4. **Discussion and conclusions:** what a future wireless system should take from your results

## 0. Setup
**First time only:** in VS Code, right-click this folder in the sidebar → *Open in Integrated Terminal*, run `bash setup_env.sh`, then click **Select Kernel** (top right) → *Jupyter Kernel…* → **ENG3014 (Python)**.

Then run the two cells below. The first loads the libraries, fixes the random seed (so your results are reproducible) and sets up report-quality plots. The second holds the measurement tools the check cells use, like the spectrum analyser on a lab bench. You don't need to edit either.

In [ ]:
# 0a. Libraries, random seed and plot style
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# One random number generator for the whole notebook. Fixing the seed means the
# "random" noise is identical on every run, so your results are reproducible.
rng = np.random.default_rng(seed=3014)

# Report-quality defaults: readable fonts, a light grid, high-resolution export.
plt.rcParams.update({
    "figure.figsize": (7, 3.2),
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# Final figures are saved next to this notebook in ./figures, ready for Overleaf.
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save a figure as PDF (vector, for LaTeX) and PNG (quick preview) in ./figures."""
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    print(f"saved figures/{name}.pdf")

print("Setup done: numpy", np.__version__)

In [ ]:
# 0b. Measurement tools used by the CHECK cells (no need to edit these)

def power(x):
    """Mean power of a signal: the average of |x|^2."""
    return np.mean(np.abs(x) ** 2)

def db(ratio):
    """Linear power ratio -> decibels, 10 log10(ratio) (L1&2 p.27)."""
    return 10 * np.log10(ratio)

def spectrum(x, fs):
    """
    One-sided power spectrum of a real signal x sampled at fs (Hz).
    Returns (f, P): bin frequencies in Hz and the power in each bin,
    scaled so that P.sum() == power(x) (Parseval's theorem).
    """
    N = len(x)
    P = np.abs(np.fft.rfft(x) / N) ** 2
    P[1:(N + 1) // 2] *= 2            # fold the negative frequencies onto the positive ones
    f = np.fft.rfftfreq(N, d=1 / fs)
    return f, P

def band_power(x, fs, f_lo, f_hi):
    """Power of x that lies between f_lo and f_hi (Hz)."""
    f, P = spectrum(x, fs)
    return P[(f >= f_lo) & (f <= f_hi)].sum()

def occupied_bandwidth(x, fs, fraction=0.99):
    """
    Width of the band holding `fraction` of the power (99% by default, the
    usual regulatory definition). Returns (B, f_lo, f_hi) in Hz.
    """
    f, P = spectrum(x, fs)
    cumulative = np.cumsum(P) / P.sum()
    f_lo = f[np.searchsorted(cumulative, (1 - fraction) / 2)]
    f_hi = f[np.searchsorted(cumulative, 1 - (1 - fraction) / 2)]
    return f_hi - f_lo, f_lo, f_hi

def psd(x, fs, n_avg=16):
    """
    Averaged power spectral density (Welch's method): chop x into overlapping
    segments, window each one (Hann, L3 p.26) and average their spectra.
    Much smoother than one long FFT, so better for plots.
    Returns (f, S): frequencies in Hz and power per Hz.
    """
    n_seg = min(len(x), 2 ** int(np.log2(max(len(x) // n_avg, 256))))
    window = np.hanning(n_seg)
    starts = range(0, len(x) - n_seg + 1, n_seg // 2)          # 50% overlap
    S = np.mean([np.abs(np.fft.rfft(window * x[s:s + n_seg])) ** 2 for s in starts], axis=0)
    S /= fs * np.sum(window ** 2)
    S[1:(n_seg + 1) // 2] *= 2
    return np.fft.rfftfreq(n_seg, d=1 / fs), S

def plot_spectrum(ax, x, fs, ref=None, **plot_kwargs):
    """
    Plot the averaged spectrum of x in dB against frequency in kHz. Levels are
    relative to `ref` (a PSD value), or to the trace's own peak if ref is None.
    """
    f, S = psd(x, fs)
    ref = S.max() if ref is None else ref
    ax.plot(f / 1e3, db(S / ref + 1e-30), **plot_kwargs)
    ax.set_xlabel("Frequency (kHz)")
    ax.set_ylabel("Power (dB)")

## 1. Message → bits
**Brief:** a message of your choice, more than 2048 bits, converted from a string to binary.

**Slides:** old L07 p.52 *Encoding information into transmittable signals* (ASCII codes and the "Hello World" example); L1&2 p.11–12 (bits, bytes, bits per character).

**Think about:** how many characters give you more than 2048 bits? 7-bit or 8-bit codes (L1&2 p.12 counts 7 bits per character)? What happens to characters outside ASCII, like é or curly quotes?

**Report notes:** *…*

In [ ]:
# 1. Your message and the string <-> binary conversion

message = """Replace this with your own message. It needs to be long enough to give more than 2048 bits."""

def text_to_bits(text):
    """Convert a string into a 1-D NumPy array of 0s and 1s."""
    # your code here
    raise NotImplementedError("text_to_bits")

def bits_to_text(bits):
    """Convert an array of 0s and 1s back into a string (the inverse of text_to_bits)."""
    # your code here
    raise NotImplementedError("bits_to_text")

In [ ]:
# CHECK 1 (run after the cell above)
bits = np.asarray(text_to_bits(message))
print(f"{len(message)} characters -> {len(bits)} bits")
assert bits.ndim == 1 and np.isin(bits, [0, 1]).all(), "bits should be a 1-D array of 0s and 1s"
assert len(bits) > 2048, f"the brief needs more than 2048 bits; you have {len(bits)}"
assert bits_to_text(bits) == message, "bits_to_text(text_to_bits(message)) doesn't give the message back"
print("First 16 bits:", bits[:16])
print("Check 1 passed")

## 2. Design choices: modulation format, carrier, baud rate, sampling
**Brief:** a modulation format of your choice. The report has to justify the format, baud rate, filter type and bandwidth.

**Slides:** L4 p.9–14 (ASK, FSK, multi-level ASK); old L06 p.33–43 (BPSK, QPSK, 8-PSK, QAM); L1&2 p.13 ($R = R_s \log_2 M$); L1&2 p.25–26 (faster symbols need more bandwidth; Nyquist $R_s \le 2B$); L3 p.25 (the Nyquist frequency is $f_s/2$).

**Constraints:**
- transmitter bandwidth $B < 0.2 f_c$ (section 4 checks this), so $R_s$ can't be too high for your $f_c$
- the signal reaches about $f_c + B/2$, so $f_s$ must sit comfortably above $2(f_c + B/2)$
- a whole number of samples per symbol ($sps = f_s / R_s$) keeps the symbol timing simple

**Practical tip:** a real 2.4 GHz carrier would need $f_s$ above 4.8 GHz, which means millions of samples for one message. Everything the brief asks for is relative ($B/f_c$, SNR), so a scaled-down carrier in the kHz range behaves the same and runs in seconds. Say so in the report.

**Think about:** which format gives a measurable BER at 6 dB, and what will it do at 18 and 24 dB? (Old L07 p.50 plots BER against SNR per bit for different $M$.)

**Report notes:** *…*

In [ ]:
# 2. System parameters: your design choices (justify each one in the report)

MOD = None      # modulation format, e.g. "ASK", "FSK", "BPSK", "QPSK", "16QAM"
M = None        # number of symbol levels (bits per symbol = log2(M))
f_c = None      # carrier frequency (Hz)
R_s = None      # symbol (baud) rate (symbols/s)
sps = None      # samples per symbol (a whole number)
f_s = None      # sample frequency (Hz); should equal sps * R_s

SNRS_DB = [6, 18, 24]   # channel SNRs from the brief

In [ ]:
# CHECK 2: parameter sanity
assert None not in (MOD, M, f_c, R_s, sps, f_s), "fill in every parameter in the cell above"
assert float(sps).is_integer(), "use a whole number of samples per symbol"
assert np.isclose(f_s, sps * R_s), "f_s should equal sps * R_s"

bits_per_symbol = np.log2(M)
n_symbols = int(np.ceil(len(bits) / bits_per_symbol))
print(f"{MOD}: {bits_per_symbol:g} bits/symbol, so bit rate R = R_s log2(M) = {R_s * bits_per_symbol:,.0f} bit/s")
print(f"Carrier {f_c:,.0f} Hz; Nyquist frequency f_s/2 = {f_s / 2:,.0f} Hz")
print(f"One transmission: {n_symbols:,} symbols, {n_symbols / R_s:.3g} s of signal, {n_symbols * int(sps):,} samples")

assert f_s > 2.2 * f_c, "Nyquist (L3 p.25): the signal reaches about f_c + B/2 < 1.1 f_c, so f_s must exceed 2.2 f_c"
if n_symbols * sps > 2_000_000:
    print("Warning: over 2 million samples per transmission, so section 7 will be slow. Fewer samples per symbol?")
print("Check 2 passed")

## 3. Modulator: bits → symbols → carrier
**Slides:** old L07 p.25 and p.29 (IQ modulation, $f(t) = A_I\cos(2\pi\nu_c t) - A_Q\sin(2\pi\nu_c t)$); old L06 p.33–43 (PSK, QPSK and QAM; the serial-to-parallel split into I and Q); L4 p.9–14 (ASK and FSK; p.10 is the AM form you coded in Lab 1).

**Steps:** group the bits into $\log_2 M$-bit symbols → map each group to an amplitude and phase (or a frequency) → hold each symbol for `sps` samples → put it on the carrier.

**Think about:** for $M > 2$, which bit pattern goes on which point, so that the most likely mistake (confusing neighbouring points) costs only one bit? What happens if `len(bits)` isn't a multiple of $\log_2 M$?

**Report notes:** *…*

In [ ]:
# 3. Modulator

def modulate(bits):
    """
    Turn a bit array into the transmitted carrier signal, sampled at f_s,
    before the output filter.
    Returns (t, tx): the sample times (s) and the real-valued signal.
    """
    # your code here
    raise NotImplementedError("modulate")

In [ ]:
# CHECK 3: look at the modulated signal in time and frequency
t, tx_raw = modulate(bits)
assert len(t) == len(tx_raw), "t and tx should be the same length"
assert np.isrealobj(tx_raw), "the transmitted signal is a real voltage: if you built it from complex numbers, take the real part"
print(f"{len(tx_raw):,} samples, mean power {power(tx_raw):.3g}")

fig, (ax_t, ax_f) = plt.subplots(2, 1, figsize=(7, 5))
n_show = int(3 * sps)                                   # the first 3 symbols
ax_t.plot(t[:n_show] * 1e3, tx_raw[:n_show], lw=0.8)
for k in (1, 2):                                        # dashed lines at the symbol boundaries
    ax_t.axvline(k * sps / f_s * 1e3, color="grey", ls="--", lw=0.8)
ax_t.set(xlabel="Time (ms)", ylabel="Amplitude", title="Modulated carrier: first 3 symbols")
plot_spectrum(ax_f, tx_raw, f_s)
ax_f.set(ylim=(-60, 3), title="Spectrum before the output filter")
plt.tight_layout()
plt.show()
# Look: where is the power, and how wide is the main lobe compared with R_s? (L4 p.15-17)

## 4. Output filter: transmitter bandwidth < 20% of $f_c$
**Brief:** an output filter limiting the transmitter bandwidth to less than 20% of the carrier frequency.

**Slides:** L4 p.15–21 (spectrum of a modulated pulse, band-pass filtering, the Butterworth response); old L04 p.27 (the output filter); L3 p.16–18 (filtering is convolution; FIR filters); Lab 1 Task 4 (a square band-pass filter, multiplying in k-space). `scipy.signal` has filter design tools if you want them.

**Think about:**
- An ideal "brick-wall" FFT filter or a realistic one (FIR or Butterworth): which could a real transmitter build, and what does each do to the pulses in time? (L1&2 p.26: squeezing the bandwidth makes neighbouring pulses overlap.)
- What does "bandwidth" mean: null-to-null, −3 dB, or 99% of the power? The check uses 99% of the power. Say which you use in the report.
- How narrow can the filter get before the receiver makes errors with no noise at all?

**Report notes:** *…*

In [ ]:
# 4. Output (transmit) filter

B_tx = None     # filter bandwidth (Hz): must be less than 0.2 * f_c

def tx_filter(x):
    """Apply the output filter to the modulated signal. Returns the filtered signal, the same length as x."""
    # your code here
    raise NotImplementedError("tx_filter")

In [ ]:
# CHECK 4: is the transmitter bandwidth under 20% of the carrier?
tx = tx_filter(tx_raw)
assert len(tx) == len(tx_raw), "keep the filtered signal the same length as the input"
B99, f_lo, f_hi = occupied_bandwidth(tx, f_s)
print(f"99% occupied bandwidth: {B99:,.0f} Hz ({f_lo:,.0f} to {f_hi:,.0f} Hz) = {100 * B99 / f_c:.1f}% of f_c")
print(f"Power kept by the filter: {100 * power(tx) / power(tx_raw):.1f}%")

fig, ax = plt.subplots()
ref = psd(tx_raw, f_s)[1].max()                         # same 0 dB reference for both traces
plot_spectrum(ax, tx_raw, f_s, ref=ref, color="lightgrey", label="before filter")
plot_spectrum(ax, tx, f_s, ref=ref, label="after filter")
ax.axvspan(0.9 * f_c / 1e3, 1.1 * f_c / 1e3, color="tab:green", alpha=0.12, label="20% of $f_c$")
ax.set(xlim=(0, 2 * f_c / 1e3), ylim=(-60, 3), title="Transmitted spectrum")
ax.legend(loc="upper right")
plt.show()

assert B99 < 0.2 * f_c, "the transmitter bandwidth must be under 20% of the carrier"
print("Check 4 passed")

## 5. AWGN channel at 6, 18 and 24 dB
**Slides:** L3 p.23 (white Gaussian noise; "the receiver's filter sets how much noise gets in"); old L05 p.20 (AWGN, Gaussian with standard deviation σ); L1&2 p.27 (6 dB is 3.98 in linear terms); L1&2 p.28 and old L07 p.49 (S/N measured in a bandwidth $B$).

**Think about:** SNR is signal power over noise power, but noise power in which bandwidth? White noise spreads evenly from 0 to $f_s/2$, while your signal only occupies $B_{tx}$. The check prints three common definitions side by side. Pick one, make your function match it, and state it in the report. The definition that matches the BER formula you compare against in section 7 (old L07 p.48–49) is the easiest to defend.

**Report notes:** *…*

In [ ]:
# 5. AWGN channel

def awgn(x, snr_db, rng=rng):
    """Return x plus white Gaussian noise, scaled so the SNR is snr_db (by your definition)."""
    # your code here
    raise NotImplementedError("awgn")

In [ ]:
# CHECK 5: measure the SNR you actually produced, three ways
R_b = R_s * np.log2(M)                                  # bit rate (bit/s)
print("target | SNR over 0 to f_s/2 | SNR inside B_tx | Eb/N0")
for snr in SNRS_DB:
    noise = awgn(tx, snr) - tx
    assert abs(np.mean(noise)) < 0.1 * np.std(noise), "the noise should have zero mean"
    in_band_noise = band_power(noise, f_s, f_c - B_tx / 2, f_c + B_tx / 2)
    N0 = in_band_noise / B_tx                           # noise power per Hz
    full_band = db(power(tx) / power(noise))
    in_band = db(power(tx) / in_band_noise)
    eb_n0 = db(power(tx) / (N0 * R_b))                  # (S/N) x B / R_b, as on old L07 p.49
    print(f"{snr:>4} dB | {full_band:16.1f} dB | {in_band:12.1f} dB | {eb_n0:5.1f} dB")
    assert min(abs(v - snr) for v in (full_band, in_band, eb_n0)) < 0.3, \
        "none of the three measures matches the target: check the dB -> linear conversion, and power vs amplitude"

# What the noise looks like on the signal
fig, axes = plt.subplots(len(SNRS_DB), 1, figsize=(7, 5.5), sharex=True)
n_show = int(3 * sps)
for ax, snr in zip(axes, SNRS_DB):
    ax.plot(t[:n_show] * 1e3, awgn(tx, snr)[:n_show], lw=0.7, label="received")
    ax.plot(t[:n_show] * 1e3, tx[:n_show], "k", lw=1, label="transmitted")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"{snr} dB", loc="left")
axes[0].legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
axes[-1].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()
print("Check 5 passed: the column that matches your target is the definition you've implemented")

## 6. Receiver: demodulate and recover the message
**Slides:** old L07 p.30–37 (IQ demodulation: mix with cos and sin, low-pass filter away the $2\nu_c$ terms, leaving $z = 0.5A_I + i\,0.5A_Q$, then the constellation); old L06 p.29–32 and p.35–37 (synchronous detection with a local oscillator; the I and Q arms; the Argand diagram); L4 p.23–25 and old L05 p.48–53 (envelope and FSK detection, the decision process).

**Steps:** remove the carrier → filter → take one sample per symbol → decide which symbol was sent → symbols → bits.

**Think about:** your filters delay the signal, so where in each symbol should you sample? Where do the decision thresholds go, and do they depend on the received amplitude?

**Report notes:** *…*

In [ ]:
# 6. Receiver

def demodulate(rx):
    """Received signal -> estimated bits (at least as many as were sent)."""
    # your code here
    raise NotImplementedError("demodulate")

In [ ]:
# CHECK 6: noiseless loopback first (no noise, so every bit should come back)
bits_hat = np.asarray(demodulate(tx))[:len(bits)]
assert len(bits_hat) == len(bits), "demodulate should return at least as many bits as were sent"
n_err = np.count_nonzero(bits_hat != bits)
print(f"Noiseless loopback: {n_err} bit errors out of {len(bits)}")
assert n_err == 0, "with no noise every bit should come back: check the sampling instant (filter delay), thresholds and bit order"
print("Recovered:", bits_to_text(bits_hat)[:100], "...")
print("Check 6 passed\n")

# Then one pass through the noisy channel at each SNR
for snr in SNRS_DB:
    bits_hat = np.asarray(demodulate(awgn(tx, snr)))[:len(bits)]
    n_err = np.count_nonzero(bits_hat != bits)
    print(f"{snr:>2} dB: {n_err} bit errors, BER = {n_err / len(bits):.2e}")

**Worth a figure:** plot what your decision stage sees at each SNR: the constellation for PSK or QAM (old L07 p.37–39), or a histogram of the decision variable for ASK or FSK. An eye diagram (old L07 p.42–45, and Lab 3) works too. If `demodulate` also keeps the values it made decisions on, this is easy.

In [ ]:
# Your constellation / eye / decision-variable plot

## 7. BER over repeated transmissions
**Brief:** analyse the bit error rate of the received signal over multiple retransmissions of the signal over the simulated channel.

**Slides:** old L07 p.47–51 (bit errors; $\mathrm{BER} = 0.5\,\mathrm{erfc}\sqrt{e_b/n_0}$, and the version with $S/N$, $B$, $\nu_s$ and $\log_2 m$; BER against SNR per bit for different $m$).

**Think about:**
- Each transmission gets fresh noise, so a single run is a random estimate. How many runs before the mean settles?
- If every run has 0 errors, the BER isn't 0. What's the smallest BER you could have measured with that many bits?
- Does your measured BER sit on the theory curve? If not, is it your SNR definition (section 5), your filter (overlapping pulses), or the formula?

`erfc` is `scipy.special.erfc` (or `math.erfc` for one number at a time).

**Report notes:** *…*

In [ ]:
# 7. BER over repeated transmissions

N_RETX = None   # transmissions per SNR

def measure_ber(snr_db, n_retx):
    """
    Send the message n_retx times through the channel at snr_db.
    Returns an array of n_retx BERs, one per transmission.
    """
    # your code here
    raise NotImplementedError("measure_ber")

In [ ]:
# CHECK 7: run the experiment and summarise it
assert N_RETX is not None, "set N_RETX in the cell above"
ber = {snr: np.asarray(measure_ber(snr, N_RETX)) for snr in SNRS_DB}
n_total = N_RETX * len(bits)
print(f"{N_RETX} transmissions x {len(bits)} bits = {n_total:,} bits per SNR")
print(f"Smallest non-zero BER you can measure: 1/{n_total:,} = {1 / n_total:.1e}")
for snr, b in ber.items():
    assert len(b) == N_RETX, "measure_ber should return one BER per transmission"
    print(f"{snr:>2} dB: mean BER {b.mean():.2e}  (runs ranged {b.min():.1e} to {b.max():.1e})")

fig, ax = plt.subplots()
for snr, b in ber.items():
    if b.mean() > 0:
        ax.semilogy(snr, b.mean(), "o", color="tab:blue")
    else:                                               # no errors: only an upper bound
        ax.semilogy(snr, 1 / n_total, "v", color="tab:red")
ax.plot([], [], "o", color="tab:blue", label="measured")
ax.plot([], [], "v", color="tab:red", label="no errors (BER below this)")
ax.set(xlabel="SNR (dB)", ylabel="Bit error rate", xticks=SNRS_DB,
       title=f"Measured BER, {N_RETX} transmissions per SNR")
ax.legend()
plt.show()
# Next: add the theoretical curve for your format to this plot (old L07 p.48-49),
# then save it with save_fig(fig, "ber_vs_snr")

## 8. Creative additions (up to 3, 10% each)
Each one needs background (slides plus a reference), an implementation, a result you can plot, and a sentence on what it means for a future system.

| Idea | Where to start |
|---|---|
| **Urban fading (Rayleigh)** | L4 p.32–34; old L08 p.23–32: slow vs fast fading, coherence time, Doppler. Old L08 p.31 gives the recipe: filter white noise with the Doppler spectrum, inverse FFT, normalise to the RMS. Old L08 p.39–41: Hata large-city path loss |
| **Hamming (7,4) error correction** | Old L07 p.53–55 and old L09 p.27–32: generator and parity-check matrices, the (8,4) extension. Is it fair to compare coded and uncoded BER at the same SNR when only 4 of every 7 bits carry the message? |
| **Automatic resending (ARQ)** | *Beyond the slides.* You need error *detection* first (a parity bit or CRC per packet); then plot throughput against SNR |
| **Dynamic modulation** | Old L07 p.50 (BER against $M$); L1&2 p.13 and p.28 (bit rate, Shannon's limit). Choose $M$ from the measured BER or SNR and plot the throughput |
| **Audio, image or video** | L1&2 p.12 (the bits needed for audio and video). A small greyscale image is easy to show received at 6, 18 and 24 dB |

In [ ]:
# 8a. Creative addition 1:

In [ ]:
# 8b. Creative addition 2:

In [ ]:
# 8c. Creative addition 3:

## 9. Figures for the report
Three pages fit about 4 or 5 figures. Save each final one with `save_fig(fig, "name")`, which writes a PDF and a PNG into `figures/` for Overleaf.

A good set: a block diagram of the simulation · the transmitted spectrum against the 20% limit · constellations (or eye diagrams) at 6, 18 and 24 dB · BER against SNR with theory · one figure per creative addition.

Before submitting: **Restart** the kernel and **Run All**, so every output comes from one clean run, and make sure every function has a docstring and comments (20% of the marks).